# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [2]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [1952, 1957, 1962, 1967, 1972, 1977, 1982, 1987, 1992, 1997, 2002, 2007]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    425.9
std      17.3     12.9  1.061579e+08     9857.5    248.3
min    1952.0     23.6  6.001100e+04      241.2      4.0


## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [ ]:
# Task 1


import pandas as pd
import plotly.graph_objects as go


continent_choice = "Asia"


plot_df = df[
    (df["continent"] == continent_choice) &
    (df["year"].isin([2002, 2007]))
].copy()

if plot_df.empty:
    raise ValueError("No data found for this continent and these years.")

# GDP per capita = GDP / population
plot_df["gdp_per_capita"] = plot_df["gdpPercap"]

fig = go.Figure()


for country in plot_df["country"].unique():
    country_data = plot_df[
        plot_df["country"] == country
    ].sort_values("year")

    if len(country_data) == 2:
        fig.add_trace(go.Scatter(
            x=country_data["gdp_per_capita"],
            y=country_data["lifeExp"],
            mode="lines",
            line=dict(color="#CCCCCC", width=0.8),
            showlegend=False,
            hoverinfo="skip"
        ))


for year, colour in [(2002, "#1f77b4"), (2007, "#e45756")]:
    year_data = plot_df[plot_df["year"] == year]

    fig.add_trace(go.Scatter(
        x=year_data["gdp_per_capita"],
        y=year_data["lifeExp"],
        mode="markers",
        name=str(year),
        marker=dict(size=9, color=colour, opacity=0.85),
        text=year_data["country"],
        customdata=year_data[["country"]],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Year: " + str(year) + "<br>"
            "GDP per capita: %{x:,.0f}<br>"
            "Life expectancy: %{y:.1f} years"
            "<extra></extra>"
        )
    ))

fig.update_layout(
    title=f"Asia: Countries Generally Moved Toward Higher Life Expectancy (2002–2007)",
    xaxis_title="GDP per capita (international dollars)",
    yaxis_title="Life expectancy (years)",
    template="plotly_white",
    width=950,
    height=600,
    legend_title="Year"
)

fig.update_xaxes(showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)

fig.show()



## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [5]:
# Task 2



import plotly.express as px
import plotly.graph_objects as go


data = df[df["year"] == 2007].copy()


highlight_continent = "Asia"
data["group"] = data["continent"].apply(
    lambda continent: continent
    if continent == highlight_continent
    else "Other continents"
)

fig = go.Figure()


other = data[data["continent"] != highlight_continent]

fig.add_trace(go.Scatter(
    x=other["gdpPercap"],
    y=other["lifeExp"],
    mode="markers",
    name="Other continents",
    text=other["country"],
    marker=dict(
        size=other["pop"],
        sizemode="area",
        sizeref=2.0 * data["pop"].max() / (35 ** 2),
        sizemin=4,
        color="#CCCCCC",
        opacity=0.65,
        line=dict(width=0.5, color="white")
    ),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "GDP per capita: %{x:,.0f}<br>"
        "Life expectancy: %{y:.1f} years"
        "<extra></extra>"
    )
))


asia = data[data["continent"] == highlight_continent]

fig.add_trace(go.Scatter(
    x=asia["gdpPercap"],
    y=asia["lifeExp"],
    mode="markers",
    name="Asia",
    text=asia["country"],
    marker=dict(
        size=asia["pop"],
        sizemode="area",
        sizeref=2.0 * data["pop"].max() / (35 ** 2),
        sizemin=4,
        color="#2878B5",
        opacity=0.8,
        line=dict(width=0.7, color="white")
    ),
    hovertemplate=(
        "<b>%{text}</b><br>"
        "GDP per capita: %{x:,.0f}<br>"
        "Life expectancy: %{y:.1f} years"
        "<extra></extra>"
    )
))

fig.add_annotation(
    x=asia["gdpPercap"].median(),
    y=asia["lifeExp"].median(),
    text="Asia: wide variation in income<br>and life expectancy",
    showarrow=True,
    arrowhead=2,
    ax=90,
    ay=-65,
    bgcolor="white",
    bordercolor="#2878B5",
    borderwidth=1
)

fig.update_layout(
    title="Asia in 2007: Countries Differ Widely in Income and Longevity",
    xaxis_title="GDP per capita (log scale)",
    yaxis_title="Life expectancy (years)",
    template="plotly_white",
    width=1000,
    height=650,
    legend_title="Continent",
    font=dict(size=12)
)

# Required logarithmic scale
fig.update_xaxes(type="log", showgrid=False, zeroline=False)
fig.update_yaxes(showgrid=False, zeroline=False)

fig.show()

